# 02 - QLoRA Fine-Tuning (Colab, T4/A100)
Fine-tunes `Qwen/Qwen2.5-1.5B-Instruct` with 4-bit QLoRA using TRL's `SFTTrainer` on the 15k-example subsample from notebook 01.

**Runtime:** GPU (T4 is sufficient at 4-bit; A100 will be faster).

Upload `data/train.jsonl` and `data/val.jsonl` from notebook 01 before running, or re-run the data-prep cells here.

In [ ]:
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
OUTPUT_DIR = 'text2sql-qwen2.5-1.5b-qlora'

In [ ]:
# If notebook 01 wasn't run in this session, upload data/train.jsonl + data/val.jsonl first.
dataset = load_dataset('json', data_files={'train': 'data/train.jsonl', 'validation': 'data/val.jsonl'})

def format_example(example):
    return {'text': example['prompt'] + example['completion']}

dataset = dataset.map(format_example)
dataset['train'][0]['text'][:500]

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    logging_steps=25,
    save_strategy='epoch',
    eval_strategy='epoch',
    bf16=True,
    max_seq_length=1024,
    dataset_text_field='text',
    packing=False,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    peft_config=lora_config,
)

In [ ]:
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

## Merge LoRA adapter into the base model (needed before GGUF conversion)

In [ ]:
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')
merged = PeftModel.from_pretrained(base, OUTPUT_DIR)
merged = merged.merge_and_unload()
merged.save_pretrained(f'{OUTPUT_DIR}-merged')
tokenizer.save_pretrained(f'{OUTPUT_DIR}-merged')

## Convert to GGUF and register with Ollama
Run in a Colab shell cell (or locally after downloading `*-merged`):
```bash
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp && pip install -r requirements.txt
python convert_hf_to_gguf.py ../text2sql-qwen2.5-1.5b-qlora-merged --outfile ../models/text2sql-qwen2.5-1.5b.gguf --outtype q4_k_m
```
Then, on the machine running Ollama (see `models/Modelfile`):
```bash
ollama create text2sql-qwen2.5-1.5b -f models/Modelfile
```